# Phase 4 Demo — Query Enrichment & Entity Graph Search

## What This Notebook Shows

This interactive notebook demonstrates how our advanced image retrieval system processes text queries:

1. **Simple text query** → A user types a description (e.g., "a girl playing violin on stage")
2. **Finding example images (seeds)** → The system finds initial matching images using CLIP
3. **Discovering related concepts** → From those examples, it identifies important entities/concepts
4. **Query enrichment** → The original query is enhanced with "Related: [concept1, concept2, ...]"
5. **Graph-based search** → A knowledge graph of concepts helps find more relevant images
6. **Comparison** → See how results differ between:
   - **CLIP-only** (traditional semantic search)
   - **CLIP + Knowledge Graph (KG)** (concept-enriched search)

### Interactive Experience

You can type your own queries and watch the full pipeline in action!

---

**⚠️ Before Running:**

Please ensure you have:
- **Activated your virtual environment** (e.g., `venv\Scripts\Activate.ps1` on Windows or `source venv/bin/activate` on Unix)
- **Selected it as the Python kernel** in VS Code (use the kernel picker in the top-right corner)
- All project dependencies installed (`pip install -r requirements.txt`)
- Phase 4 data artifacts available (entity graph, embeddings, indices)

## Setup & Imports

In this step we:
- Import necessary libraries
- Load Phase 4 configuration from `configs/entity_graph.yaml`
- Prepare the components: text encoder (CLIP), entity graph, and metadata

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any

import torch
import numpy as np

# Project imports
from src.graph.config import (
    load_entity_graph_config,
    get_entity_graph_config,
    get_query_enrichment_config,
    get_graph_search_config,
    get_fusion_config,
)
from src.graph.graph_search import (
    enrich_query,
    graph_search,
    image_scores_dict,
)
from src.graph.build_entity_graph import load_entity_graph
from src.retrieval.bi_encoder import BiEncoder
from src.retrieval.hybrid_search import HybridSearchEngine

# Load Phase 4 configuration
CFG_PATH = "configs/entity_graph.yaml"

raw_cfg = load_entity_graph_config(CFG_PATH)
entity_cfg = get_entity_graph_config(raw_cfg)
qe_cfg = get_query_enrichment_config(raw_cfg)
gs_cfg = get_graph_search_config(raw_cfg)
fusion_cfg = get_fusion_config(raw_cfg)

print("✓ Loaded Phase 4 configuration")
print(f"  • Entity graph path: {entity_cfg['entity_graph_path']}")
print(f"  • Query enrichment: K_seed_raw = {qe_cfg['K_seed_raw']}, M_enrich = {qe_cfg['M_enrich']}")
print(f"  • Graph search: H_max = {gs_cfg['H_max']}, decay = {gs_cfg['decay']}")
print(f"  • Fusion default mode: {fusion_cfg.get('default_mode')}")

## Load Models, Graph, and Metadata

Now we load:
1. **The text encoder (CLIP)** — To understand your query and each concept (entity)
2. **The entity graph** — A "map of concepts" showing how ideas connect to each other
3. **Metadata** — Information about which images and captions each concept appears in

In [ ]:
# Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Instantiate text/image encoder (CLIP)
print("\nLoading CLIP encoder...")
bi_encoder = BiEncoder(device=device)
print("✓ CLIP encoder loaded")

# Instantiate the full HybridSearchEngine
print("\nInitializing hybrid search engine...")
hybrid_engine = HybridSearchEngine(
    bi_encoder=bi_encoder,
    config_path="configs/hybrid_config.yaml",
    device=device,
)
print("✓ Hybrid search engine initialized")

In [ ]:
# Load the entity graph
graph_path = entity_cfg["entity_graph_path"]
print(f"Loading entity graph from: {graph_path}")
graph = load_entity_graph(graph_path)

# The graph is a PyTorch Geometric HeteroData object
# Extract statistics about the graph structure
num_entities = graph["entity"].x.size(0)
num_sem_edges = graph["entity", "sem", "entity"].edge_index.size(1) if ("entity", "sem", "entity") in graph.edge_types else 0
num_cooc_edges = graph["entity", "cooc", "entity"].edge_index.size(1) if ("entity", "cooc", "entity") in graph.edge_types else 0

print(f"✓ Entity graph loaded")
print(f"  • Entities: {num_entities}")
print(f"  • Semantic edges: {num_sem_edges}")
print(f"  • Co-occurrence edges: {num_cooc_edges}")

# Load entity context and vocabulary/metadata
print("\nLoading entity metadata...")
with open(entity_cfg["context_path"], "r") as f:
    entity_context: Dict[str, Any] = json.load(f)

with open(entity_cfg["vocab_path"], "r") as f:
    entity_vocab: Dict[str, Any] = json.load(f)

with open(entity_cfg["entity_meta_path"], "r") as f:
    entity_meta: Dict[str, Any] = json.load(f)

print(f"✓ Loaded entity context, vocab, and meta")
print(f"  • Total entities: {len(entity_vocab)}")

# Load image database if available
image_db_path = Path("data/entities/image_db.json")
if image_db_path.exists():
    with open(image_db_path, "r") as f:
        image_db = json.load(f)
    print(f"✓ Loaded image database: {len(image_db)} images")
else:
    image_db = {}
    print("⚠ No image_db.json found; will show image IDs only")

## Helper Functions for Pretty Printing

These functions transform technical outputs into human-readable tables and summaries.

In [ ]:
def pretty_print_seeds(candidates: List[Tuple[str, float]], image_db: Dict, max_rows: int = 5) -> None:
    """
    Print top seed images in a friendly table format.
    
    Args:
        candidates: List of (image_id, score) tuples
        image_db: Dictionary mapping image_id to metadata
        max_rows: Maximum number of rows to display
    """
    print(f"\n{'Rank':<6} {'Image ID':<15} {'Score':<8} Caption")
    print("-" * 80)
    
    for rank, (img_id, score) in enumerate(candidates[:max_rows], 1):
        caption = "N/A"
        if image_db and img_id in image_db:
            captions = image_db[img_id].get("captions", [])
            if captions:
                caption = captions[0][:60] + "..." if len(captions[0]) > 60 else captions[0]
        
        print(f"{rank:<6} {img_id:<15} {score:<8.4f} {caption}")


def pretty_print_entities(enrichment_result, top_n: int = 10) -> None:
    """
    Print discovered entities (concepts) in a friendly table.
    
    Args:
        enrichment_result: Result object from enrich_query()
        top_n: Number of top entities to display
    """
    entity_ids = enrichment_result.entity_ids[:top_n]
    entity_names = enrichment_result.entity_names[:top_n]
    entity_scores = enrichment_result.entity_scores[:top_n]
    
    print(f"\n{'Rank':<6} {'Concept':<30} {'Score':<10}")
    print("-" * 80)
    
    for rank, (ent_id, name, score) in enumerate(zip(entity_ids, entity_names, entity_scores), 1):
        # Truncate long entity names
        display_name = name[:28] + ".." if len(name) > 30 else name
        print(f"{rank:<6} {display_name:<30} {score:<10.4f}")


def pretty_print_graph_results(kg_result, image_db: Dict, top_n: int = 5) -> None:
    """
    Print top images from graph search results.
    
    Args:
        kg_result: Result object from graph_search()
        image_db: Dictionary mapping image_id to metadata
        top_n: Number of top images to display
    """
    kg_scores = image_scores_dict(kg_result)
    
    # Sort by score descending
    sorted_items = sorted(kg_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    print(f"\n{'Rank':<6} {'Image ID':<15} {'KG Score':<10} Caption")
    print("-" * 80)
    
    for rank, (img_id, score) in enumerate(sorted_items, 1):
        caption = "N/A"
        if image_db and img_id in image_db:
            captions = image_db[img_id].get("captions", [])
            if captions:
                caption = captions[0][:60] + "..." if len(captions[0]) > 60 else captions[0]
        
        print(f"{rank:<6} {img_id:<15} {score:<10.4f} {caption}")


def show_config_summary(qe_cfg: Dict[str, Any], gs_cfg: Dict[str, Any]) -> None:
    """
    Display configuration settings in human-readable format.
    """
    print("\n" + "=" * 80)
    print("CONFIGURATION SUMMARY")
    print("=" * 80)
    
    print("\n📊 Query Enrichment Settings:")
    print(f"  • Number of seed images used to discover concepts: K_seed_raw = {qe_cfg['K_seed_raw']}")
    print(f"  • Number of concepts attached to the query: M_enrich = {qe_cfg['M_enrich']}")
    print(f"  • Similarity threshold for entity relevance: sigma_sim = {qe_cfg.get('sigma_sim', 'N/A')}")
    
    print("\n🔍 Graph Search Settings:")
    print(f"  • Maximum hops in the concept graph: H_max = {gs_cfg['H_max']}")
    print(f"  • Maximum nodes processed: B = {gs_cfg['B']}")
    print(f"  • Score decay per hop: decay = {gs_cfg['decay']}")
    print(f"  • Minimum edge weight: min_weight = {gs_cfg.get('min_weight', 'N/A')}")
    
    print("\n" + "=" * 80)

print("✓ Helper functions loaded")

In [ ]:
# Display current configuration
show_config_summary(qe_cfg, gs_cfg)

## Walkthrough: From Query → Enrichment → Graph Search

Let's follow one example query step by step:

1. **CLIP finds example images (seeds)** — Initial semantic search results
2. **System discovers important concepts** — Entities extracted from seed images
3. **Enriched query is built** — Original query + discovered concepts
4. **Graph walk finds boosted images** — Knowledge graph traversal scores images

In [ ]:
# Pick a demo query
demo_query = "a girl playing violin on stage"
print(f"\n🎯 Demo Query: \"{demo_query}\"")
print("\nLet's see how the system processes this query...")

### Step 1: CLIP-Only Seeds (Stage 1 Retrieval)

First, we use CLIP to find images that semantically match the query.  
These become our "seed" examples.

In [ ]:
K1 = 64  # Number of candidates to retrieve in Stage 1

print(f"Retrieving top {K1} seed images using CLIP...")
candidates: List[Tuple[str, float]] = hybrid_engine._stage1_retrieve(
    query=demo_query,
    k1=K1,
    show_progress=False,
)

print(f"\n✓ Found {len(candidates)} candidate images")
print("\nTop 5 example images (seeds) from CLIP:")
pretty_print_seeds(candidates, image_db, max_rows=5)

print(
    "\n💡 These are the first images our system thinks might match your query. "
    "We will use them as examples (seeds) to discover related concepts."
)

### Step 2: Query Enrichment with Discovered Concepts

Now we analyze the seed images to discover important concepts (entities).  
The system:
- Looks at what entities appear in the seed images
- Measures how relevant each entity is to the original query
- Selects the top M entities to "enrich" the query

In [ ]:
# Prepare seeds for enrichment
K_seed_raw = qe_cfg["K_seed_raw"]
seeds_for_enrichment = candidates[:K_seed_raw]

print(f"Using top {K_seed_raw} seeds for concept discovery...\n")

# Run query enrichment
enrichment_result = enrich_query(
    query=demo_query,
    seeds=seeds_for_enrichment,
    encoders=bi_encoder,
    entity_context=entity_context,
    cfg=raw_cfg,
)

print("=" * 80)
print("QUERY ENRICHMENT RESULT")
print("=" * 80)

print("\n📝 Original query:")
print(f"   {enrichment_result.original_query}")

print("\n✨ Enriched query:")
print(f"   {enrichment_result.enriched_query}")

print("\n🔑 Top discovered concepts:")
pretty_print_entities(enrichment_result, top_n=10)

print(
    "\n💡 The system looked at the example images and found concepts that appear "
    "frequently and are semantically close to your query. It then attached those "
    "concepts to your query as 'Related: [concept1, concept2, ...]'."
)

### Step 3: Graph Search Using Seeds + Enriched Query

Now we walk the entity knowledge graph:
- Start from seed entities (found in seed images)
- Expand through connected entities (up to H_max hops)
- Score images based on which concepts they contain
- Images with strong combinations of relevant concepts get boosted

In [ ]:
print("Running graph search...\n")

# Perform graph search
kg_result = graph_search(
    query=demo_query,
    graph=graph,
    encoders=bi_encoder,
    cfg=raw_cfg,
    seeds=seeds_for_enrichment,
)

kg_scores = image_scores_dict(kg_result)

print("=" * 80)
print("GRAPH SEARCH SUMMARY")
print("=" * 80)

print(f"\n📍 Seed entities: {len(kg_result.seed_entity_ids)}")
print(f"🔍 Total entities visited: {len(kg_result.entity_ids)}")
print(f"🖼️  Images with non-zero KG score: {len(kg_scores)}")
print(f"⏱️  Runtime: {kg_result.runtime_ms:.2f} ms")

print("\n🏆 Top images by knowledge graph score:")
pretty_print_graph_results(kg_result, image_db, top_n=5)

print(
    "\n💡 These images are boosted because they contain strong combinations of the "
    "discovered concepts, even if their captions don't exactly match your words. "
    "This is the power of the knowledge graph!"
)

### Step 4: Compare CLIP-Only vs CLIP + Knowledge Graph

Let's see the difference side by side:

In [ ]:
print("\n" + "=" * 80)
print("COMPARISON: CLIP-ONLY vs CLIP + KNOWLEDGE GRAPH")
print("=" * 80)

print("\n🔵 CLIP-ONLY (Stage 1) — Top 5:")
print("   (Pure semantic similarity between query and image embeddings)")
pretty_print_seeds(candidates, image_db, max_rows=5)

print("\n🟢 CLIP + KNOWLEDGE GRAPH — Top 5:")
print("   (Boosted by concept combinations discovered from the graph)")
pretty_print_graph_results(kg_result, image_db, top_n=5)

print(
    "\n💡 Notice how the Knowledge Graph may surface different images that have "
    "relevant concepts, potentially improving recall for complex queries."
)

## Try Your Own Query!

Now it's your turn! Type any query (in English) and see:
1. The first example images we find
2. The concepts we discover
3. The enriched query
4. The top images scored by the concept graph

**Example queries to try:**
- "a dog running in the park"
- "children playing soccer"
- "a sunset over the ocean"
- "a person reading a book in a library"
- "a red car on a highway"

In [ ]:
def run_query_demo(query: str, k1: int = 64, top_show: int = 5) -> None:
    """
    Run the complete Phase 4 pipeline for any query.
    
    Args:
        query: Text query to search for
        k1: Number of Stage 1 candidates to retrieve
        top_show: Number of top results to display
    """
    print("\n" + "=" * 80)
    print("PHASE 4 PIPELINE DEMO")
    print("=" * 80)
    print(f"\n🎯 QUERY: \"{query}\"")
    print("=" * 80)

    # ========== STEP 1: Stage 1 Seeds ==========
    print("\n" + "─" * 80)
    print("STEP 1 — Finding Example Images (Seeds) with CLIP")
    print("─" * 80)
    
    candidates = hybrid_engine._stage1_retrieve(
        query=query,
        k1=k1,
        show_progress=False,
    )
    
    print(f"\n✓ Retrieved {len(candidates)} candidates")
    print(f"\nTop {top_show} seed images:")
    pretty_print_seeds(candidates, image_db, max_rows=top_show)

    # ========== STEP 2: Query Enrichment ==========
    print("\n" + "─" * 80)
    print("STEP 2 — Discovering Concepts & Enriching Query")
    print("─" * 80)
    
    K_seed_raw = qe_cfg["K_seed_raw"]
    seeds_for_enrichment = candidates[:K_seed_raw]
    
    print(f"\nAnalyzing top {K_seed_raw} seeds for concept discovery...")
    
    enrichment_result = enrich_query(
        query=query,
        seeds=seeds_for_enrichment,
        encoders=bi_encoder,
        entity_context=entity_context,
        cfg=raw_cfg,
    )
    
    print("\n📝 Original query:")
    print(f"   {enrichment_result.original_query}")
    
    print("\n✨ Enriched query:")
    print(f"   {enrichment_result.enriched_query}")
    
    print("\n🔑 Discovered concepts:")
    pretty_print_entities(enrichment_result, top_n=10)

    # ========== STEP 3: Graph Search ==========
    print("\n" + "─" * 80)
    print("STEP 3 — Walking the Concept Graph")
    print("─" * 80)
    
    print("\nTraversing entity knowledge graph...")
    
    kg_result = graph_search(
        query=query,
        graph=graph,
        encoders=bi_encoder,
        cfg=raw_cfg,
        seeds=seeds_for_enrichment,
    )
    
    kg_scores = image_scores_dict(kg_result)
    
    print(f"\n✓ Graph search complete")
    print(f"  • Seed entities: {len(kg_result.seed_entity_ids)}")
    print(f"  • Entities visited: {len(kg_result.entity_ids)}")
    print(f"  • Images scored: {len(kg_scores)}")
    print(f"  • Runtime: {kg_result.runtime_ms:.2f} ms")

    # ========== STEP 4: Results ==========
    print("\n" + "─" * 80)
    print("STEP 4 — Final Results")
    print("─" * 80)
    
    print(f"\n🏆 Top {top_show} images by concept graph score:")
    pretty_print_graph_results(kg_result, image_db, top_n=top_show)
    
    # ========== Summary ==========
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(
        "\n🎓 What just happened:\n"
        "   1. We started from your text query\n"
        "   2. Found example images using CLIP\n"
        "   3. Discovered important concepts from those examples\n"
        "   4. Used a graph of concepts to find images with relevant concept combinations\n"
        "   5. Scored images based on both semantic similarity AND concept co-occurrence\n"
        "\n💡 This is how knowledge graphs enhance traditional semantic search!"
    )
    print("=" * 80)

print("✓ Interactive demo function loaded")

### Interactive Query Interface

Use the text box below to enter your query, then click "Run Demo" to see the full pipeline in action!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create UI components
query_box = widgets.Text(
    value="a girl playing violin on stage",
    description="Query:",
    layout=widgets.Layout(width="80%"),
    style={'description_width': '60px'},
)

run_button = widgets.Button(
    description="🚀 Run Demo",
    button_style="primary",
    tooltip="Click to run the Phase 4 pipeline",
    layout=widgets.Layout(width="150px"),
)

output = widgets.Output()

# Button click handler
def on_run_clicked(b):
    with output:
        clear_output(wait=True)
        if query_box.value.strip():
            run_query_demo(query_box.value)
        else:
            print("⚠️ Please enter a query!")

run_button.on_click(on_run_clicked)

# Display UI
print("Enter your query and click the button to see the Phase 4 pipeline in action!\n")
display(query_box, run_button, output)

## Optional: Mode Comparison (CLIP-only vs CLIP+KG vs Full Hybrid)

If you want to compare different search modes, run the cell below.  
This shows how results differ across:
- **clip_only**: Pure CLIP semantic search
- **clip_kg**: CLIP + Knowledge Graph enrichment
- **full**: CLIP + BLIP-2 + Knowledge Graph (if cross-encoder is available)

In [ ]:
# Mode comparison (optional)
comparison_query = "a girl playing violin on stage"
modes = ["clip_only", "clip_kg", "full"]

print("\n" + "=" * 80)
print("MODE COMPARISON")
print("=" * 80)
print(f"\nQuery: \"{comparison_query}\"\n")

for mode in modes:
    print("\n" + "─" * 80)
    print(f"Mode: {mode.upper()}")
    print("─" * 80)
    
    try:
        results = hybrid_engine.text_to_image_hybrid_search(
            query=comparison_query,
            k1=64,
            k2=5,
            mode=mode,
            show_progress=False,
        )
        
        print(f"\nTop 5 results for mode '{mode}':")
        pretty_print_seeds(results, image_db, max_rows=5)
        
    except Exception as e:
        print(f"⚠️ Error running mode '{mode}': {e}")
        print("   (This mode may require additional components not yet configured)")

print("\n" + "=" * 80)
print(
    "💡 Notice how different modes may surface different images.\n"
    "   - 'clip_only': Fast, semantic similarity only\n"
    "   - 'clip_kg': Enhanced with knowledge graph concept boosting\n"
    "   - 'full': Maximum quality with CLIP + BLIP-2 cross-encoder + KG"
)
print("=" * 80)

## Conclusion

🎉 **Congratulations!** You've explored the Phase 4 pipeline:

✅ **Query Processing** — From simple text to enriched queries  
✅ **Concept Discovery** — Automatic entity extraction from seed images  
✅ **Graph-Enhanced Search** — Leveraging knowledge graphs for better retrieval  
✅ **Interactive Exploration** — Try your own queries and see results instantly  

### Key Takeaways

1. **Knowledge graphs enhance semantic search** by understanding concept relationships
2. **Query enrichment** helps bridge the semantic gap between user intent and image content
3. **Multi-hop graph traversal** discovers relevant images through concept co-occurrence
4. **Hybrid approaches** combine multiple signals (CLIP, BLIP-2, KG) for robust retrieval

### Next Steps

- Explore other notebooks: `08_hybrid_search_evaluation.ipynb`, `09_inspect_entity_graph.ipynb`
- Tune configuration parameters in `configs/entity_graph.yaml`
- Experiment with different queries and analyze the enrichment results
- Try building your own entity graph with custom entities

---

**Questions or feedback?** Check the project README or open an issue on GitHub!